Задача: Расчет загрузки самолетов. 
1. Определить среднюю загрузку самолетов (процент занятых мест) для каждого рейса - только успешно совершившего перелет (status = 'Arrived') и вылетевших в сентябре 2016 года.

Используем таблицы:
flights для получения информации о рейсах.
aircrafts для получения общей вместимости самолета.
seats для подсчета общего числа мест.
boarding_passes для подсчета фактически занятых мест.


2. Провести оптимизацию скрипта по необходимости и показать разницу в затраченных ресурсов и преимуществе.

In [ ]:
explain analyze
with sold_seats as (
	select 
		f.flight_id ,
		f.aircraft_code, 
		count(*) as sold
	from flights f
	join ticket_flights tf on tf.flight_id =f.flight_id 
	where f.status = 'Arrived' and 
			f.scheduled_arrival >= '2016-09-01' and f.scheduled_arrival < '2016-10-01' 
	group by 1, 2
	),
	total as (
		select
			s.aircraft_code,
			count(*) as cnt
		from seats s 
		group by 1
	)
select 
	ss.flight_id,
	ss.aircraft_code,
	ss.sold,
	t.cnt,
	round((ss.sold:: numeric / t.cnt * 100),2 )
from sold_seats ss
left join total t on ss.aircraft_code = t.aircraft_code 



			
-- Execution Time: 370.077 ms
-- seats 144 k
-- flights 69 m
-- ticket_flights 871 

In [ ]:


create index idx_ticket_flights_flight_id
	on bookings.ticket_flights(flight_id)

create index idx_flights_status_scheduled_arrival 
	on bookings.flights(status, scheduled_arrival)
	

-- Execution Time: 76.149 ms
-- seats 144 k
-- flights 72 m
-- ticket_flights 929 

После провидения оптимизации запроса, скорость запроса увеличилась, но также незначительно увеличился объем данных. 
Также при использовании функции extract на колонке с датой, увеличиваеться время выполнения запроса.